# Module 20 - Capstone: a tiny ChatGPT

Use this notebook after `tests/test_assistant.py` is passing. The goal is not to introduce a new model algorithm; it is to assemble the pieces you built in Modules 16-19 into one usable assistant: backend, tools, retrieval, agent loop, conversation memory, eval gate, and CLI transcript.

The notebook starts with deterministic fake-backend checks so you can debug architecture without model randomness. The later cells optionally load ProdLM for live assistant runs.

1. Read the lesson page (`docs/modules/20-capstone.md`).
2. Open this notebook with `./notebook.sh 20`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import io
import json
import subprocess
import sys

from IPython.display import Markdown, display

from g2c.eval.match import numeric_match
from g2c.assistant import (
    Assistant,
    AssistantConfig,
    AssistantTurn,
    Conversation,
    EvalCase,
    run_cli,
    run_evaluation,
)
from g2c.inference import Backend, BackendInfo, InferenceResult, load_selected_backend
from g2c.notebook_extras.sampling import printable
from g2c.rag import Chunk, DenseRetriever, HashEmbedder, NumpyVectorStore, chunk_text
from g2c.tools import ToolRegistry, make_calculator, make_read_file

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
print(repo_root)

Run the assistant tests before proceeding. In the clean scaffold this cell should fail until you implement the Module 20 TODOs in `g2c/assistant/`.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_assistant.py", "-q"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 20 assistant tests are not passing yet."

## Display helpers

In [ ]:
def short(text: Any, limit: int = 180) -> str:
    rendered = printable(str(text)).replace("\n", "\\n")
    if len(rendered) <= limit:
        return rendered
    return rendered[: limit - 3] + "..."


def show_conversation(conversation: Conversation) -> None:
    if not conversation.messages:
        display(Markdown("*(conversation is empty)*"))
        return
    rows = ["| # | role | content |", "|---:|---|---|"]
    for i, message in enumerate(conversation.messages, start=1):
        rows.append(f"| {i} | `{message.role}` | {short(message.content, 220)} |")
    display(Markdown("\n".join(rows)))


def show_agent_steps(turn: AssistantTurn) -> None:
    rows = [
        "| step | thought | action | args | observation | error | final | parse error |",
        "|---:|---|---|---|---|---|---|---|",
    ]
    for i, step in enumerate(turn.agent_run.steps, start=1):
        action = step.action.tool if step.action else ""
        args = json.dumps(step.action.arguments) if step.action else ""
        observation = step.observation.output if step.observation else ""
        is_error = "yes" if step.observation and step.observation.is_error else "no" if step.observation else ""
        rows.append(
            "| "
            + " | ".join(
                [
                    str(i),
                    short(step.thought, 120),
                    action,
                    short(args, 120),
                    short(observation, 160),
                    is_error,
                    short(step.final_answer or "", 160),
                    short(step.parse_error or "", 120),
                ]
            )
            + " |"
        )
    display(Markdown("\n".join(rows)))


def show_turn(turn: AssistantTurn) -> None:
    display(Markdown(f"**Final answer:** {turn.final_answer or '(none)'}"))
    display(Markdown(f"**Stopped reason:** `{turn.agent_run.stopped_reason}`"))
    display(Markdown(f"**Metadata:** `{json.dumps(turn.metadata, sort_keys=True)}`"))
    if turn.retrieved_context:
        display(Markdown("**Retrieved context**"))
        print(turn.retrieved_context)
    display(Markdown("**Contextualized message**"))
    print(turn.contextualized_message)
    display(Markdown("**Agent steps**"))
    show_agent_steps(turn)


def show_eval_report(report) -> None:
    rows = ["| case | passed | final answer | failure |", "|---|---:|---|---|"]
    for result in report.results:
        rows.append(
            f"| {result.case.name} | {result.passed} | "
            f"{short(result.final_answer or '', 160)} | "
            f"{short(result.failure_reason or '', 160)} |"
        )
    display(Markdown(report.summary()))
    display(Markdown("\n".join(rows)))

## Deterministic fake backend

A real model makes this module interesting, but fake completions make the architecture testable. The fake backend below returns pre-recorded ReAct completions in order and records every prompt it receives.

In [ ]:
class FakeBackend(Backend):
    def __init__(self, completions, *, info: BackendInfo | None = None) -> None:
        self._completions = list(completions)
        self._info = info or BackendInfo(name="fake", model_id="fake-model")
        self.calls: list[dict[str, Any]] = []

    @property
    def info(self) -> BackendInfo:
        return self._info

    def complete(
        self,
        prompt: str,
        *,
        max_new_tokens: int = 128,
        temperature: float = 1.0,
        top_k: int | None = None,
        top_p: float | None = None,
    ) -> InferenceResult:
        if not self._completions:
            raise AssertionError(f"FakeBackend exhausted. Prompt was:\n{prompt}")
        completion = self._completions.pop(0)
        self.calls.append(
            {
                "prompt": prompt,
                "max_new_tokens": max_new_tokens,
                "temperature": temperature,
                "top_k": top_k,
                "top_p": top_p,
            }
        )
        return InferenceResult(
            prompt=prompt,
            completion=completion,
            prompt_tokens=len(prompt.split()),
            completion_tokens=len(completion.split()),
            latency_ms=1.0,
            backend=self._info,
        )


def planning_completion() -> str:
    return "Goal: solve the task\n1. inspect the request\n2. use tools if useful\n3. answer"


def final_answer_completion(answer: str) -> str:
    return f"Thought: I now know the final answer.\nFinal Answer: {answer}"


def action_completion(tool: str, args_json: str, thought: str = "I should use a tool.") -> str:
    return f"Thought: {thought}\nAction: {tool}\nAction Input: {args_json}"

## Build local capstone resources

The capstone assistant needs tools and, optionally, documents to retrieve from. This cell creates a tiny sandboxed directory under `data/work/module20/` and builds a registry with calculator and read-file tools.

In [ ]:
module20_dir = repo_root / "data" / "work" / "module20"
module20_dir.mkdir(parents=True, exist_ok=True)

(module20_dir / "numbers.txt").write_text("3\n7\n10\n20\n", encoding="utf-8")
(module20_dir / "assistant_policy.md").write_text(
    "The capstone assistant should cite retrieved context when it uses documents.\n"
    "It should use the calculator for exact arithmetic rather than mental math.\n",
    encoding="utf-8",
)
(module20_dir / "course_notes.md").write_text(
    "Module 16 provides the inference Backend.\n"
    "Module 17 provides retrieval over document chunks.\n"
    "Module 18 provides tools and tool dispatch.\n"
    "Module 19 provides the ReAct agent loop.\n"
    "Module 20 composes these pieces into an assistant.\n",
    encoding="utf-8",
)

registry = ToolRegistry([
    make_calculator(),
    make_read_file(root=module20_dir),
])
print(registry)

## Exercise 1 - Conversation memory

`Conversation` is the new Module 20 primitive. It is not the same as the Module 19 scratchpad: conversation survives across `Assistant.chat(...)` calls; scratchpad exists only inside one agent run.

In [ ]:
conversation = Conversation(max_messages=4)
conversation.add_user("My name is Ada.")
conversation.add_assistant("Nice to meet you, Ada.")
conversation.add_user("What module built the agent loop?")

print(conversation.format_for_prompt())
show_conversation(conversation)

In [ ]:
truncated = Conversation(max_messages=2)
truncated.add_user("old user message")
truncated.add_assistant("old assistant reply")
truncated.add_user("new user message")
print(truncated.format_for_prompt())
assert "old user message" not in truncated.format_for_prompt()
assert "new user message" in truncated.format_for_prompt()

In [ ]:
"Question: The Module 19 scratchpad and the Module 20 Conversation both hold history. What breaks if you merge them into one structure?"
"Answer: "

## Exercise 2 - Minimal assistant turn

Start with no RAG and no tools used. This checks the outer shell: `Assistant.chat` should build a contextualized message, run the agent, store the final answer, and append to conversation history.

In [ ]:
config = AssistantConfig(plan=False, rag_enabled=False, max_steps=4, temperature=0.0, use_native=False)
backend = FakeBackend([final_answer_completion("Hello from the capstone assistant.")])
assistant = Assistant(backend, registry, config=config)

turn = assistant.chat("Say hello in one sentence.")
show_turn(turn)
show_conversation(assistant.conversation)

## Exercise 3 - Multi-turn history

The second turn should see the first turn as `Previous conversation:`. The current question should appear once, outside that history block.

In [ ]:
backend = FakeBackend([
    final_answer_completion("The agent loop was Module 19."),
    final_answer_completion("You asked about Module 19."),
])
assistant = Assistant(backend, registry, config=config)

first = assistant.chat("Which module built the agent loop?")
second = assistant.chat("What did I just ask about?")

show_turn(second)
assert "Previous conversation" in second.contextualized_message
assert "Which module built the agent loop?" in second.contextualized_message
assert second.contextualized_message.count("What did I just ask about?") == 1

In [ ]:
"Question: Why must the conversation history be rendered before the current user message is appended? What symptom shows up in the contextualized message if the order is wrong?"
"Answer: "

## Exercise 4 - Tools through the assistant

The assistant delegates tool use to the Module 19 agent, which delegates dispatch to Module 18. This run should call the calculator, observe the result, then answer.

In [ ]:
tool_backend = FakeBackend([
    action_completion("calculator", '{"expression": "(1847 * 29) - 138"}', "I should compute this exactly."),
    final_answer_completion("The final integer is 53425."),
])
tool_assistant = Assistant(tool_backend, registry, config=config)

tool_turn = tool_assistant.chat("Use the calculator to compute (1847 * 29) - 138.")
show_turn(tool_turn)
assert any(step.action and step.action.tool == "calculator" for step in tool_turn.agent_run.steps)

## Exercise 5 - File + calculator workflow

This is the smallest useful agentic pattern: read external state, transform it with a tool, then answer.

In [ ]:
file_backend = FakeBackend([
    action_completion("read_file", '{"path": "numbers.txt"}', "I need the file contents first."),
    action_completion("calculator", '{"expression": "(3 + 7 + 10 + 20) / 4"}', "I should compute the average."),
    final_answer_completion("The average is 10."),
])
file_assistant = Assistant(file_backend, registry, config=config)

file_turn = file_assistant.chat("Read numbers.txt and tell me the average.")
show_turn(file_turn)
assert file_turn.final_answer == "The average is 10."

## Exercise 6 - Build a tiny retriever

Module 20 uses RAG by prefixing retrieved chunks into the message handed to the agent. Here we index the small Module 20 sandbox files with the Module 17 hash embedder.

In [ ]:
rag_chunks: list[Chunk] = []
for path in sorted(module20_dir.glob("*.md")):
    text = path.read_text(encoding="utf-8")
    rag_chunks.extend(chunk_text(text, chunk_size=240, chunk_overlap=40, source=path.name))

embedder = HashEmbedder(dim=512, ngram_range=(3, 5), seed=20)
vectors = embedder.embed([chunk.text for chunk in rag_chunks])
store = NumpyVectorStore(dim=embedder.dim)
store.add(rag_chunks, vectors)
retriever = DenseRetriever(embedder, store)

retrieved = retriever.retrieve("Which modules make up the assistant stack?", k=3)
for item in retrieved:
    print(f"rank={item.rank} score={item.score:.3f} source={item.chunk.source}")
    print(short(item.chunk.text, 240))

## Exercise 7 - RAG through the assistant

In this mode, `Assistant.chat` retrieves context before calling the agent. The agent does not need to remember to call a retrieval tool; the relevant document snippets are already in the contextualized message.

In [ ]:
rag_config = AssistantConfig(plan=False, rag_enabled=True, rag_k=2, max_steps=4, temperature=0.0, use_native=False)
rag_backend = FakeBackend([
    final_answer_completion("Modules 16, 17, 18, and 19 feed the Module 20 assistant.")
])
rag_assistant = Assistant(rag_backend, registry, config=rag_config, retriever=retriever)

rag_turn = rag_assistant.chat("Which earlier modules feed the capstone assistant?")
show_turn(rag_turn)
assert rag_turn.metadata["rag_fired"] is True
assert "Context from documents" in rag_turn.contextualized_message

In [ ]:
"Question: This assistant prefixes retrieved chunks into the message instead of giving the agent a retrieval tool. What is the tradeoff, and when should the model decide for itself when to search?"
"Answer: "

## Exercise 8 - Regression eval gate

A capstone assistant needs a small eval suite that catches integration regressions: Did RAG fire? Did the calculator get called? Did the assistant produce a final answer?

In [ ]:
eval_backend = FakeBackend([
    action_completion("calculator", '{"expression": "21 * 2"}', "I should compute this exactly."),
    final_answer_completion("42"),
    final_answer_completion("Module 20 composes backend, retrieval, tools, and the agent loop."),
])
eval_assistant = Assistant(
    eval_backend,
    registry,
    config=AssistantConfig(plan=False, rag_enabled=True, rag_k=2, max_steps=4, temperature=0.0, use_native=False),
    retriever=retriever,
)
cases = [
    EvalCase(
        name="calculator_path",
        question="Use the calculator for 21 * 2.",
        expected_answer="42",
        matcher=numeric_match,
        expected_tool="calculator",
        rag=False,
    ),
    EvalCase(
        name="capstone_context",
        question="What does Module 20 compose?",
        expected_answer=["backend", "backends"],
        rag=True,
    ),
]
report = run_evaluation(eval_assistant, cases, reset_each=True)
show_eval_report(report)
assert report.pass_rate == 1.0

In [ ]:
"Question: The eval cases assert which tool fired and whether RAG fired, not just the answer text. Why should Module 20 evals test integration rather than the base model's factual recall?"
"Answer: "

## Exercise 9 - Optional live assistant with model selection

The default live backend is ProdLM. To test a course-trained model instead, set `MODEL_SELECTION = "course"` for the strongest course artifact, or set it to a base artifact name like `"TinyLLM-30M"`; the loader will prefer `-DPO`, then `-SFT`, then the base artifact.

In [ ]:
MODEL_SELECTION = "ProdLM"  # "ProdLM", "course", or an artifact base/name such as "TinyLLM-30M"
PRODLM_MODEL_ID = None  # optional Ollama tag override when MODEL_SELECTION == "ProdLM"
LIVE_DEVICE = "auto"
LIVE_TORCH_DTYPE = "float16"

live_backend = None
try:
    live_backend = load_selected_backend(
        MODEL_SELECTION,
        repo_root=repo_root,
        prodlm_model_id=PRODLM_MODEL_ID,
        device=LIVE_DEVICE,
        torch_dtype=LIVE_TORCH_DTYPE,
        required=False,
    )
    if live_backend is None:
        print("No live backend loaded. Run ./prodlm.sh or choose an available artifact.")
    else:
        print("loaded:", live_backend.info)
except Exception as exc:
    print(f"Live backend unavailable: {type(exc).__name__}: {exc}")

In [ ]:
live_assistant = None
if live_backend is None:
    print("Skipping live assistant construction.")
else:
    live_config = AssistantConfig(
        name="module20-live",
        plan=True,
        rag_enabled=True,
        rag_k=2,
        max_steps=6,
        max_new_tokens=384,
        temperature=0.1,
        scratchpad_max_chars=4_000,
        max_history_messages=10,
    )
    live_assistant = Assistant(
        live_backend,
        registry,
        config=live_config,
        retriever=retriever,
    )
    print(live_assistant)

In [ ]:
if live_assistant is None:
    print("No live assistant loaded.")
else:
    live_turn = live_assistant.chat(
        "Use the calculator to compute (1847 * 29) - 138, then give the final integer.",
        use_rag=False,
    )
    show_turn(live_turn)

In [ ]:
if live_assistant is None:
    print("No live assistant loaded.")
else:
    live_rag_turn = live_assistant.chat(
        "According to the local course notes, which modules feed the Module 20 assistant?",
        use_rag=True,
    )
    show_turn(live_rag_turn)
    show_conversation(live_assistant.conversation)

In [ ]:
"Question: If you swapped MODEL_SELECTION to a course-trained artifact, where did it stop being viable as a chat backend? Give one concrete task where ProdLM succeeds and the from-scratch model cannot."
"Answer: "

## Exercise 9b - Channel comparison

`AssistantConfig.use_native` controls which agent channel the assistant uses under the hood: `True` (the default) wires a `NativeAgent` — structured tool calling via `backend.chat_with_tools`. `False` falls back to the ReAct `Agent` from Module 19. Both expose the same `assistant.chat(...)` surface; only the wire format between the assistant and the model changes.

Module 19's Exercise 12b made the same point at the agent layer; this is the Module 20 version at the assistant layer. Same task, same backend, same registry — different channel. Expect the native channel to fail less often on small models, because the model has been post-trained for that wire format. ReAct fails more colorfully (bad parses, mid-prose JSON, runaway loops), which is informative for the post-mortem.

In [ ]:
if live_backend is None:
    print("Skipping channel comparison: No live backend is loaded.")
else:
    channel_question = "Use the calculator to compute (1847 * 29) - 138. Then give the final integer."

    print("Native channel (Module 20 default):")
    native_assistant = Assistant(
        live_backend,
        registry,
        config=AssistantConfig(
            name="module20-native",
            plan=False,
            rag_enabled=False,
            max_steps=6,
            max_new_tokens=384,
            temperature=0.0,
            use_native=True,
        ),
    )
    native_turn = native_assistant.chat(channel_question)
    show_turn(native_turn)

    print("\nReAct channel (use_native=False):")
    react_assistant = Assistant(
        live_backend,
        registry,
        config=AssistantConfig(
            name="module20-react",
            plan=False,
            rag_enabled=False,
            max_steps=6,
            max_new_tokens=384,
            temperature=0.0,
            use_native=False,
        ),
    )
    react_turn = react_assistant.chat(channel_question)
    show_turn(react_turn)

In [ ]:
"Question: On the calculator task, which channel produced the cleaner turn: native or ReAct? Name the failure mode you saw on the worse channel."
"Answer: "

## Exercise 10 - CLI transcript

The CLI is deliberately small. It is useful because it gives you a flat transcript you can save and inspect after failures.

In [ ]:
cli_backend = FakeBackend([final_answer_completion("CLI response")])
cli_assistant = Assistant(cli_backend, registry, config=config)
transcript_path = repo_root / "data" / "work" / "module20" / "transcript.json"
inp = io.StringIO(f"hello from the CLI\n/history\n/save {transcript_path}\n\n")
out = io.StringIO()
run_cli(cli_assistant, inp=inp, out=out)

print(out.getvalue())
print("saved:", transcript_path.exists(), transcript_path)

## Exercise 11 - Failure-mode catalog

Run a few live prompts and collect at least five failures. The important skill is localization: did the problem come from retrieval, tool selection, argument formatting, agent loop control, backend capability, or conversation history?

In [ ]:
failure_modes = [
    {
        "name": "example_unknown_tool",
        "symptom": "The model used an operator or prose as an Action name.",
        "likely_layer": "agent prompt / model format-following",
        "mitigation": "Make tool names and parameter schemas more explicit; lower temperature; add an eval case.",
    },
    {
        "name": "example_retrieval_miss",
        "symptom": "The answer ignores the document that contains the needed fact.",
        "likely_layer": "retriever / chunking / embedding",
        "mitigation": "Inspect retrieved chunks, adjust chunk size, add semantic embeddings or hybrid retrieval.",
    },
]
display(Markdown("| failure | likely layer | mitigation |\n|---|---|---|"))
for item in failure_modes:
    display(Markdown(f"| {item['name']} | {item['likely_layer']} | {item['mitigation']} |"))

In [ ]:
"Question: Pick one live failure you observed. Which layer did you localize it to - retrieval, tool selection, argument formatting, agent loop control, backend capability, or conversation history - and what evidence pointed there?"
"Answer: "

## Exercise 12 - Post-mortem scaffold

The course deliverable is the post-mortem. Write it after you have run the assistant, broken it, fixed at least one thing, and rerun the eval gate.

In [ ]:
postmortem_template = """# Capstone post-mortem

## Assistant configuration

- Backend:
- Tools:
- Retriever:
- Conversation/history limits:
- Eval suite:

## What each layer contributed

- Inference backend:
- Retrieval:
- Tools:
- Agent loop:
- Conversation memory:
- Eval gate:

## Failure modes

1.
2.
3.
4.
5.

## What I would improve next

- 
"""
print(postmortem_template)
print("Suggested path:", repo_root / "docs" / "capstone-postmortem.md")

In [ ]:
"Question: Across the whole stack, which layer contributed most to answer quality and which was the weakest link? What would you improve first?"
"Answer: "

## Deliverable checklist

- `Conversation.format_for_prompt` and `Assistant.chat` are implemented.
- `tests/test_assistant.py` passes.
- The notebook includes at least one deterministic tool run, one RAG run, and one eval report.
- Optional: live model run with a saved transcript.
- `docs/capstone-failure-modes.md` records at least five observed failures.
- `docs/capstone-postmortem.md` explains how the full stack behaved and where it broke.

When complete, ask a coding agent to grade your Module 20 notebook. Partial work is fine: the agent should grade answered questions and implemented sections, then skip blank prompts.